In [2]:
"""
Top Crypto — Up to 5 Years Daily OHLCV
Sources  : Binance → KuCoin → Kraken (fallback chain)
Coin list: CoinGecko /coins/markets pages 1–20 (~2500 coins)
Incremental: only fetches NEW dates for existing coins
NO indicators — run crypto_indicator_processor.py separately after this.
"""

import requests
import pandas as pd
import time
import os
import json
from datetime import datetime, timedelta, timezone
from typing import Optional

# ── Config ────────────────────────────────────────────────────────────────────
OUTPUT_DIR  = "crypto_data"
MASTER_FILE = "crypto_all_5yr.csv"
CHECKPOINT  = ".checkpoint.json"

YEARS_HISTORY = 5
PAGES         = 10       # 10 × 250 = up to 2500 coins

STABLECOINS = {
    "USDT","USDC","BUSD","DAI","TUSD","USDP","GUSD","FRAX",
    "LUSD","USDD","FDUSD","PYUSD","USDE","USDX","SUSD",
    "USTC","UST","EURS","XAUT","PAXG","USDN","HUSD","RSV",
}

BINANCE_URL = "https://api.binance.com/api/v3/klines"
KUCOIN_URL  = "https://api.kucoin.com/api/v1/market/candles"
KRAKEN_URL  = "https://api.kraken.com/0/public/OHLC"
CG_URL      = "https://api.coingecko.com/api/v3"

BINANCE_QUOTES = ["USDT", "BUSD", "FDUSD", "BTC", "ETH", "BNB", "TUSD"]
KUCOIN_QUOTES  = ["USDT", "BTC", "ETH", "BNB"]
KRAKEN_QUOTES  = ["USD",  "USDT", "BTC", "ETH"]


# ═══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def now_utc() -> datetime:
    return datetime.now(timezone.utc)

def ts_ms(dt: datetime) -> int:
    return int(dt.timestamp() * 1000)

def get(url: str, params: dict = {}, max_retries: int = 5, base_delay: float = 2.0):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 200:
                return r.json()
            elif r.status_code == 429:
                wait = base_delay * (2 ** attempt)
                print(f"    ⏳ Rate limited — waiting {wait:.0f}s")
                time.sleep(wait)
            elif r.status_code >= 500:
                wait = base_delay * (2 ** attempt)
                print(f"    ⚠ Server {r.status_code} — waiting {wait:.0f}s")
                time.sleep(wait)
            else:
                return None
        except Exception:
            time.sleep(base_delay * (2 ** attempt))
    return None

def load_checkpoint() -> dict:
    if os.path.exists(CHECKPOINT):
        try:
            data = json.load(open(CHECKPOINT))
            if isinstance(data, list):
                return {"done": data, "last_date": {}, "pair": {}, "exchange": {}}
            data.setdefault("exchange", {})
            return data
        except:
            pass
    return {"done": [], "last_date": {}, "pair": {}, "exchange": {}}

def save_checkpoint(cp: dict):
    json.dump(cp, open(CHECKPOINT, "w"), indent=2)


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1 — COINGECKO COIN LIST
# ═══════════════════════════════════════════════════════════════════════════════

def fetch_coin_list(pages: int = PAGES) -> pd.DataFrame:
    print(f"[1/3] Fetching top coins from CoinGecko ({pages} pages × 250)...")
    coins = []
    for page in range(1, pages + 1):
        data = get(f"{CG_URL}/coins/markets", params={
            "vs_currency": "usd",
            "order":       "market_cap_desc",
            "per_page":    250,
            "page":        page,
            "sparkline":   "false",
        })
        if data:
            coins.extend(data)
            print(f"  Page {page}/{pages} — {len(data)} coins fetched")
        else:
            print(f"  Page {page}/{pages} — failed, stopping here")
            break

        # CoinGecko free tier: burst of ~4 pages then throttles hard
        # Every 4 pages → 60s cooldown, otherwise 8s between pages
        if page % 4 == 0:
            print(f"  ⏸  Cooling down 60s after page {page} (CoinGecko rate limit)...")
            time.sleep(60)
        else:
            time.sleep(8)

    rows = []
    for c in coins:
        sym = c.get("symbol", "").upper()
        rows.append({
            "coin_id":            c.get("id"),
            "symbol":             sym,
            "name":               c.get("name"),
            "market_cap_usd":     c.get("market_cap"),
            "volume_24h_usd":     c.get("total_volume"),
            "change_24h_pct":     c.get("price_change_percentage_24h"),
            "ath":                c.get("ath"),
            "atl":                c.get("atl"),
            "circulating_supply": c.get("circulating_supply"),
            "max_supply":         c.get("max_supply"),
            "is_stablecoin":      sym in STABLECOINS,
        })

    df = pd.DataFrame(rows).drop_duplicates("coin_id").reset_index(drop=True)

    def cap_tier(mc):
        if pd.isna(mc):  return "unknown"
        if mc >= 10e9:   return "large"
        if mc >= 1e9:    return "mid"
        if mc >= 100e6:  return "small"
        return                  "micro"

    df["cap_tier"] = df["market_cap_usd"].apply(cap_tier)
    print(f"  → {len(df)} unique coins  ({df['is_stablecoin'].sum()} stablecoins flagged)")
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2 — EXCHANGE FETCHERS (raw OHLCV only, no indicators)
# ═══════════════════════════════════════════════════════════════════════════════

# ── Binance ───────────────────────────────────────────────────────────────────

def binance_find_pair(symbol: str) -> Optional[str]:
    for quote in BINANCE_QUOTES:
        if symbol == quote:
            continue
        pair = f"{symbol}{quote}"
        r = requests.get(
            BINANCE_URL,
            params={"symbol": pair, "interval": "1d", "limit": 1},
            timeout=10
        )
        if r.status_code == 200 and r.json():
            return pair
    return None

def binance_fetch_full(pair: str, start_dt: datetime, end_dt: datetime) -> Optional[pd.DataFrame]:
    """Paginated — Binance max 1000 candles per call, 5yr needs ~2 calls."""
    all_rows      = []
    current_start = ts_ms(start_dt)
    end_ms        = ts_ms(end_dt)

    while current_start < end_ms:
        data = get(BINANCE_URL, params={
            "symbol":    pair,
            "interval":  "1d",
            "startTime": current_start,
            "endTime":   end_ms,
            "limit":     1000,
        })
        if not data:
            break
        all_rows.extend(data)
        last_close_time = int(data[-1][6])
        current_start   = last_close_time + 1
        if len(data) < 1000:
            break
        time.sleep(0.1)

    if not all_rows:
        return None

    df = pd.DataFrame(all_rows, columns=[
        "open_time","open","high","low","close","volume",
        "close_time","quote_volume","trades",
        "taker_buy_base","taker_buy_quote","ignore"
    ])
    df["date"]       = pd.to_datetime(df["open_time"], unit="ms").dt.normalize()
    df["open"]       = df["open"].astype(float)
    df["high"]       = df["high"].astype(float)
    df["low"]        = df["low"].astype(float)
    df["close"]      = df["close"].astype(float)
    df["volume_usd"] = df["quote_volume"].astype(float)
    df["pair"]       = pair
    df["exchange"]   = "binance"

    return df[["date","open","high","low","close","volume_usd","pair","exchange"]] \
             .drop_duplicates("date").sort_values("date").reset_index(drop=True)


# ── KuCoin ────────────────────────────────────────────────────────────────────

def kucoin_find_pair(symbol: str) -> Optional[str]:
    for quote in KUCOIN_QUOTES:
        if symbol == quote:
            continue
        pair   = f"{symbol}-{quote}"
        end_ts = int(now_utc().timestamp())
        r = requests.get(KUCOIN_URL, params={
            "symbol":  pair,
            "type":    "1day",
            "startAt": end_ts - 86400,
            "endAt":   end_ts,
        }, timeout=10)
        try:
            if r.status_code == 200 and r.json().get("data"):
                return pair
        except:
            pass
    return None

def kucoin_fetch_full(pair: str, start_dt: datetime, end_dt: datetime) -> Optional[pd.DataFrame]:
    """Paginated — KuCoin max 1500 candles per call."""
    all_rows      = []
    current_start = int(start_dt.timestamp())
    end_ts        = int(end_dt.timestamp())
    CHUNK         = 1500 * 86400

    while current_start < end_ts:
        chunk_end = min(current_start + CHUNK, end_ts)
        data = get(KUCOIN_URL, params={
            "symbol":  pair,
            "type":    "1day",
            "startAt": current_start,
            "endAt":   chunk_end,
        })
        if not data or not data.get("data"):
            break
        all_rows.extend(data["data"])
        current_start = chunk_end + 1
        if len(data["data"]) < 100:
            break
        time.sleep(0.2)

    if not all_rows:
        return None

    df = pd.DataFrame(all_rows, columns=[
        "open_time","open","close","high","low","volume","turnover"
    ])
    df["date"]       = pd.to_datetime(df["open_time"].astype(int), unit="s").dt.normalize()
    df["open"]       = df["open"].astype(float)
    df["high"]       = df["high"].astype(float)
    df["low"]        = df["low"].astype(float)
    df["close"]      = df["close"].astype(float)
    df["volume_usd"] = df["turnover"].astype(float)
    df["pair"]       = pair
    df["exchange"]   = "kucoin"

    return df[["date","open","high","low","close","volume_usd","pair","exchange"]] \
             .drop_duplicates("date").sort_values("date").reset_index(drop=True)


# ── Kraken ────────────────────────────────────────────────────────────────────

def kraken_find_pair(symbol: str) -> Optional[str]:
    remap = {"BTC": "XBT", "DOGE": "XDG"}
    sym   = remap.get(symbol, symbol)
    for quote in KRAKEN_QUOTES:
        q_remap = {"USD": "ZUSD", "BTC": "XBT"}
        q = q_remap.get(quote, quote)
        for pair in [f"{sym}{q}", f"X{sym}Z{q}", f"{sym}/{quote}"]:
            r = requests.get(KRAKEN_URL, params={
                "pair":     pair,
                "interval": 1440,
                "since":    int(now_utc().timestamp()) - 86400
            }, timeout=10)
            try:
                result = r.json()
                if result.get("result") and not result.get("error"):
                    return pair
            except:
                pass
    return None

def kraken_fetch_full(pair: str, start_dt: datetime, end_dt: datetime) -> Optional[pd.DataFrame]:
    """Paginated — Kraken max 720 candles per call, 5yr needs ~3 calls."""
    all_rows = []
    since    = int(start_dt.timestamp())
    end_ts   = int(end_dt.timestamp())

    while since < end_ts:
        data = get(KRAKEN_URL, params={
            "pair":     pair,
            "interval": 1440,
            "since":    since,
        })
        if not data or data.get("error") or not data.get("result"):
            break
        result_key = [k for k in data["result"].keys() if k != "last"][0]
        candles    = data["result"][result_key]
        last_ts    = int(data["result"]["last"])
        all_rows.extend(candles)
        if last_ts <= since or len(candles) < 10:
            break
        since = last_ts
        time.sleep(0.5)

    if not all_rows:
        return None

    df = pd.DataFrame(all_rows, columns=[
        "open_time","open","high","low","close","vwap","volume","count"
    ])
    df["date"]       = pd.to_datetime(df["open_time"].astype(int), unit="s").dt.normalize()
    df["open"]       = df["open"].astype(float)
    df["high"]       = df["high"].astype(float)
    df["low"]        = df["low"].astype(float)
    df["close"]      = df["close"].astype(float)
    df["volume_usd"] = df["volume"].astype(float) * df["vwap"].astype(float)
    df["pair"]       = pair
    df["exchange"]   = "kraken"

    return df[["date","open","high","low","close","volume_usd","pair","exchange"]] \
             .drop_duplicates("date").sort_values("date").reset_index(drop=True)


# ── Fallback chain ────────────────────────────────────────────────────────────

def fetch_ohlcv_any_exchange(
    symbol: str,
    start_dt: datetime,
    end_dt: datetime,
    known_pair: str = None,
    known_exchange: str = None
) -> tuple[Optional[pd.DataFrame], Optional[str], Optional[str]]:
    """
    Binance → KuCoin → Kraken.
    If known_pair + known_exchange cached, goes straight there (faster updates).
    Returns (df, pair, exchange) or (None, None, None).
    """
    # Fast path — we already know where this coin lives
    if known_pair and known_exchange:
        fetchers = {
            "binance": binance_fetch_full,
            "kucoin":  kucoin_fetch_full,
            "kraken":  kraken_fetch_full,
        }
        fn = fetchers.get(known_exchange)
        if fn:
            df = fn(known_pair, start_dt, end_dt)
            if df is not None:
                return df, known_pair, known_exchange

    # Discovery path — try all three exchanges
    for find_fn, fetch_fn, exch_name in [
        (binance_find_pair, binance_fetch_full, "binance"),
        (kucoin_find_pair,  kucoin_fetch_full,  "kucoin"),
        (kraken_find_pair,  kraken_fetch_full,  "kraken"),
    ]:
        pair = find_fn(symbol)
        if pair:
            df = fetch_fn(pair, start_dt, end_dt)
            if df is not None and len(df) >= 30:
                return df, pair, exch_name
        time.sleep(0.2)

    return None, None, None


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 65)
    print(f"  Multi-Exchange Crypto Fetcher — {YEARS_HISTORY} Years Raw OHLCV")
    print(f"  Exchanges : Binance → KuCoin → Kraken (fallback chain)")
    print("=" * 65)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    cp            = load_checkpoint()
    done          = set(cp.get("done", []))
    last_date_map = cp.get("last_date", {})
    pair_map      = cp.get("pair", {})
    exchange_map  = cp.get("exchange", {})

    meta = fetch_coin_list(pages=PAGES)

    start_full = now_utc() - timedelta(days=365 * YEARS_HISTORY)
    end_dt     = now_utc().replace(hour=0, minute=0, second=0, microsecond=0)

    new_coins      = [r for _, r in meta.iterrows() if r["coin_id"] not in done]
    existing_coins = [r for _, r in meta.iterrows() if r["coin_id"] in done]

    print(f"\n  New coins to fetch (full {YEARS_HISTORY}yr) : {len(new_coins)}")
    print(f"  Existing coins to update            : {len(existing_coins)}")

    stats      = {"success": 0, "updated": 0, "skipped": 0, "not_found": 0}
    start_time = time.time()

    META_COLS = [
        "coin_id","symbol","name","market_cap_usd","cap_tier",
        "ath","atl","circulating_supply","max_supply",
        "change_24h_pct","volume_24h_usd","is_stablecoin"
    ]

    # ── A: Update existing coins (only new dates) ─────────────────────────────
    print(f"\n[A] Updating existing coins with new dates...")
    for row in existing_coins:
        coin_id    = row["coin_id"]
        symbol     = row["symbol"]
        fpath      = os.path.join(OUTPUT_DIR, f"{coin_id}.csv")
        known_pair = pair_map.get(coin_id)
        known_exch = exchange_map.get(coin_id)

        if not os.path.exists(fpath):
            new_coins.append(row)
            continue

        existing_df = pd.read_csv(fpath, parse_dates=["date"])
        last_dt     = existing_df["date"].max()
        gap_days    = (end_dt.date() - last_dt.date()).days

        if gap_days <= 1:
            stats["skipped"] += 1
            continue

        update_start = last_dt + timedelta(days=1)
        df_new, pair, exch = fetch_ohlcv_any_exchange(
            symbol, update_start, end_dt, known_pair, known_exch
        )

        if df_new is not None and len(df_new) > 0:
            df_new = df_new[df_new["date"] < pd.Timestamp(end_dt.date())]
            for col in META_COLS:
                if col in row.index:
                    df_new[col] = row[col]

            combined = (
                pd.concat([existing_df, df_new], ignore_index=True)
                .drop_duplicates("date")
                .sort_values("date")
                .reset_index(drop=True)
            )
            # Save raw — no indicators
            combined.to_csv(fpath, index=False)

            pair_map[coin_id]      = pair
            exchange_map[coin_id]  = exch
            last_date_map[coin_id] = combined["date"].max().strftime("%Y-%m-%d")
            stats["updated"] += 1
            print(f"  ↑ {symbol:12s} | +{len(df_new):3d} new rows | {exch}")
        else:
            stats["skipped"] += 1

        cp = {"done": list(done), "last_date": last_date_map,
              "pair": pair_map, "exchange": exchange_map}
        save_checkpoint(cp)
        time.sleep(0.15)

    # ── B: Fetch new coins (full history) ─────────────────────────────────────
    print(f"\n[B] Fetching new coins (up to {YEARS_HISTORY} years)...")
    total = len(meta)

    for idx, row in enumerate(new_coins):
        coin_id = row["coin_id"]
        symbol  = row["symbol"]
        rank    = meta[meta["coin_id"] == coin_id].index[0] + 1 \
                  if coin_id in meta["coin_id"].values else "?"

        df_ohlcv, pair, exch = fetch_ohlcv_any_exchange(symbol, start_full, end_dt)

        if df_ohlcv is None or len(df_ohlcv) < 10:
            print(f"  ✗ [{rank}/{total}] {symbol:12s} — not found on any exchange")
            stats["not_found"] += 1
        else:
            df_ohlcv = df_ohlcv[df_ohlcv["date"] < pd.Timestamp(end_dt.date())]

            # Attach metadata — no indicators computed here
            for col in META_COLS:
                if col in row.index:
                    df_ohlcv[col] = row[col]

            fpath = os.path.join(OUTPUT_DIR, f"{coin_id}.csv")
            df_ohlcv.to_csv(fpath, index=False)

            pair_map[coin_id]      = pair
            exchange_map[coin_id]  = exch
            last_date_map[coin_id] = df_ohlcv["date"].max().strftime("%Y-%m-%d")
            stats["success"] += 1

            elapsed = time.time() - start_time
            done_n  = stats["success"] + stats["not_found"] + stats["skipped"] + idx
            rate    = done_n / elapsed if elapsed > 0 else 1
            eta     = int((len(new_coins) - idx - 1) / rate / 60) if rate else "?"

            print(
                f"  ✓ [{rank}/{total}] {symbol:12s} | "
                f"{len(df_ohlcv):4d} days | {exch:8s} | {pair:15s} | ETA ~{eta}min"
            )

        done.add(coin_id)
        cp = {"done": list(done), "last_date": last_date_map,
              "pair": pair_map, "exchange": exchange_map}
        save_checkpoint(cp)
        time.sleep(0.15)

    # ── Merge all per-coin files into one master CSV ───────────────────────────
    print(f"\n[3/3] Merging all per-coin files into master CSV...")
    frames = []
    for coin_id in done:
        fpath = os.path.join(OUTPUT_DIR, f"{coin_id}.csv")
        if os.path.exists(fpath):
            try:
                frames.append(pd.read_csv(fpath, parse_dates=["date"]))
            except:
                pass

    if frames:
        master = (
            pd.concat(frames, ignore_index=True)
            .sort_values(["coin_id", "date"])
            .reset_index(drop=True)
        )
        master.to_csv(MASTER_FILE, index=False)

        exch_counts = {}
        for e in exchange_map.values():
            exch_counts[e] = exch_counts.get(e, 0) + 1

        print(f"\n{'='*65}")
        print("  FETCH COMPLETE")
        print(f"{'='*65}")
        print(f"  Coins fetched successfully : {stats['success']}")
        print(f"  Coins updated (new rows)   : {stats['updated']}")
        print(f"  Coins already up to date   : {stats['skipped']}")
        print(f"  Coins not found anywhere   : {stats['not_found']}")
        print(f"  Exchange breakdown         : {exch_counts}")
        print(f"  Total rows in master       : {len(master):,}")
        print(f"  Date range                 : {master['date'].min().date()} → {master['date'].max().date()}")
        print(f"  Unique coins               : {master['coin_id'].nunique()}")
        print(f"\n  Raw OHLCV saved → {MASTER_FILE}")
        print(f"  Per-coin files  → {OUTPUT_DIR}/")
        print(f"\n  ⚡ Next step: run crypto_indicator_processor.py")
        print(f"     INPUT_CSV = '{MASTER_FILE}'")

    print("\n✅ Done. Run again anytime — only new dates will be fetched.")


if __name__ == "__main__":
    main()

  Multi-Exchange Crypto Fetcher — 5 Years Raw OHLCV
  Exchanges : Binance → KuCoin → Kraken (fallback chain)
  Indicators: ❌ SKIPPED — run crypto_indicator_processor.py after
[1/3] Fetching top coins from CoinGecko (10 pages × 250)...
  Page 1/10 — 250 coins fetched
  Page 2/10 — 250 coins fetched
  Page 3/10 — 250 coins fetched
  Page 4/10 — 250 coins fetched
  ⏸  Cooling down 60s after page 4 (CoinGecko rate limit)...
  Page 5/10 — 250 coins fetched
  Page 6/10 — 250 coins fetched
    ⏳ Rate limited — waiting 2s
    ⏳ Rate limited — waiting 4s
    ⏳ Rate limited — waiting 8s
    ⏳ Rate limited — waiting 16s
    ⏳ Rate limited — waiting 32s
  Page 7/10 — failed, stopping here
  → 1499 unique coins  (26 stablecoins flagged)

  New coins to fetch (full 5yr) : 1499
  Existing coins to update            : 0

[A] Updating existing coins with new dates...

[B] Fetching new coins (up to 5 years)...
  ✓ [1/1499] BTC          | 1824 days | binance  | BTCUSDT         | ETA ~20min
  ✓ [2/1499] E